# ISTAT parser walkthrough

This notebook is the practical guide for parsing official ISTAT input-output releases in MARIO.


## What this notebook covers

- where the official ISTAT releases come from;
- when to pass one workbook, one extracted release directory, or one zip archive;
- how `IOT` and `SUT` workflows differ;
- how `iot_mode=`, `level=`, `price=`, and `valuation=` are used;
- how `download=True` works with the official ISTAT release pages;
- which caveats matter for this parser.


## Relevant source page

- Official ISTAT tag page: [ISTAT sistema input-output](https://www.istat.it/tag/sistema-input-output/)

This parser targets the official ISTAT release bundles, not arbitrary spreadsheets with a similar shape.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_istat(...)`

The same function supports:

- `IOT` parsing;
- `SUT` parsing;
- local parsing from one workbook, one extracted release directory, or one release zip;
- automatic download of the official release through `download=True`.


## Key arguments

The key public arguments are:

- `path`: local workbook, extracted release directory, release zip, or cache directory;
- `year`: reference year to select from the ISTAT release;
- `table`: choose `"IOT"` or `"SUT"`;
- `iot_mode`: for `IOT`, either `"product"` or `"industry"`;
- `level`: for `SUT`, either `"63"` or `"20"`;
- `price`: for `SUT`, either `"current"` or `"pyp"`;
- `valuation`: for `SUT`, either `"basic"` or `"purchaser"`;
- `download`: when `True`, MARIO downloads the official release before parsing it;
- `edition` and `page_url`: downloader selectors for the official ISTAT release page.


## `IOT` versus `SUT`

Use `table="IOT"` when you want the symmetric representation with `Z`, `Y`, and `V`.

Use `table="SUT"` when you want the native split structure with `S`, `U`, `Yc`, `Ya`, `Va`, and `Vc`.

For `IOT`, the key selector is `iot_mode=`. For `SUT`, the key selectors are `level=`, `price=`, and `valuation=`.


## Workbook, extracted directory, or zip archive

For `IOT`, passing the workbook directly is often the simplest workflow.

For `SUT`, passing the extracted release directory or the release zip is usually more convenient, because MARIO resolves the required workbook triplet from the official file names.


## Download workflow

If you already have the official release locally, point the parser to the workbook, directory, or zip file.

If you want MARIO to fetch the official release first, use `download=True` together with a cache directory in `path`.


In [1]:
import mario

## Parse one local `IOT` workbook

This is the simplest `IOT` workflow. If you haven't downloaded it, you can indicate a folder path and download=True


In [3]:
db = mario.parse_istat(
    path="/path/to/istat_cache",
    year=2022,
    table="IOT",
    iot_mode="product",
    download=True,
    edition="latest",
)


INFO Parser: reading ISTAT workbook SIMM_TOT_63PxP.xlsx sheet STOTPP_2022.
INFO Parser: ISTAT IOT payload ready with shapes Z=(63, 63), Y=(63, 7), V=(10, 63).
INFO Metadata: initialized.


## Parse one `SUT` release

The `SUT` workflow needs the structural selectors explicitly.


In [8]:
db = mario.parse_istat(
    path="/path/to/istat_release",
    year=2022,
    table="SUT",
    level="63",
    price="current",
    valuation="basic",
    download=True,
    edition="latest",
)

db

INFO Parser: reading ISTAT workbook SUPPLY_63B.xlsx sheet sup22.
INFO Parser: reading ISTAT workbook USEPB_63B.xlsx sheet uspb22.
INFO Parser: reading ISTAT workbook IMPORT_63B.xlsx sheet imprt22.
INFO Parser: ISTAT SUT payload ready with shapes S=(63, 63), U=(63, 63), Yc=(63, 7), Va=(12, 63), Vc=(13, 63).
INFO Metadata: initialized.


name = ISTAT SUT 2022 63 current basic
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 63
Commodity = 63
Factor of production = 13
Satellite account = 1
Consumption category = 7
Region = 1

## First inspection

Once the release is parsed, the normal MARIO inspection helpers work exactly as for the other parsers.


In [9]:
db.table_type

'SUT'

In [12]:
db.activities

['Produzioni vegetali e animali, caccia e servizi connessi',
 'Silvicoltura e utilizzo di aree forestali',
 'Pesca e acquicoltura',
 'Attività estrattiva',
 'Industrie alimentari, delle bevande e del tabacco',
 'Industrie tessili, confezione di articoli di abbigliamento e di articoli in pelle e simili',
 'Industria del legno e dei prodotti in legno e sughero, esclusi i mobili; fabbricazione di articoli in paglia e materiali da intreccio',
 'Fabbricazione di carta e di prodotti di carta',
 'Stampa e riproduzione su supporti registrati',
 'Fabbricazione di coke e prodotti derivanti dalla raffinazione del petrolio',
 'Fabbricazione di prodotti chimici',
 'Fabbricazione di prodotti farmaceutici di base e di preparati farmaceutici',
 'Fabbricazione di articoli in gomma e materie plastiche',
 'Fabbricazione di altri prodotti della lavorazione di minerali non metalliferi',
 'Attività metallurgiche',
 'Fabbricazione di prodotti in metallo, esclusi macchinari e attrezzature',
 'Fabbricazione di

In [11]:
db.units

{'Activity':                                                                unit
Produzioni vegetali e animali, caccia e servizi...  Milioni di euro
Silvicoltura e utilizzo di aree forestali           Milioni di euro
Pesca e acquicoltura                                Milioni di euro
Attività estrattiva                                 Milioni di euro
Industrie alimentari, delle bevande e del tabacco   Milioni di euro
...                                                             ...
Attività sportive, di intrattenimento e di dive...  Milioni di euro
Attività di organizzazioni associative              Milioni di euro
Riparazione di computer e di beni per uso perso...  Milioni di euro
Altre attività di servizi personali                 Milioni di euro
Attività di famiglie e convivenze come datori d...  Milioni di euro

[63 rows x 1 columns], 'Commodity':                                                                unit
Prodotti dell’agricoltura e della caccia e rela...  Milioni di eur

## Caveats

- the parser expects the official ISTAT workbook and release naming conventions;
- `SUT` parsing has more structural selectors than `IOT`, so explicit arguments are usually the better choice;
- `download=True` is optional, but it is often the cleanest workflow when you want a stable local archive of the release.
